# Generalization to multi-dim, multi-batch 

Now that we can safely do things with $n_{samples}>1$, now let's do $n>1$.

We are primarily interested in making the constraint's total derivative matrices first in an explicit way before moving to the matrix-free versions. 

## Imports and model definition

We use a residual structure and typical Jax construction. The main difference is that the naming of the layers is changed for lexcicographic sorting. 

In [1]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
from flax import linen as nn
import numpy as np

# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [2]:
class MLP(nn.Module):
    """
    We first deal with something that's not exactly MLP, but close enough
    """
    num_units: int
    
    def setup(self):
        self.dense1 = nn.Dense(self.num_units)
        self.dense2 = nn.Dense(self.num_units)
    
    def __call__(self, x):
        f = self.dense1(x)
        # f = nn.leaky_relu(f)
        f = nn.tanh(f)
        f = self.dense2(f)
        # f = nn.tanh(f)
        # x = self.dense2(x)
        return f
    

class SimpleMLP(nn.Module):
    num_layers: int
    num_units: int
    num_classes: int

    def setup(self):
        # Create a list of Dense layers
        self.layers = (
            *[
                MLP(self.num_units, name=f"layer_{i:02}") for i in range(self.num_layers - 1)
            ], 
            MLP(self.num_units, name=f"layer_{self.num_layers-1:02}") # For now, do all the same dimensions 
        )
        
    def __call__(self, x):
        # Store input to match the notes
        self.sow('intermediates',  f'layer_{0}_output', x)
        
        # Pass the input through each layer
        for i, layer in enumerate(self.layers):
            x = layer(x)
            
            # Store input to match the notes
            self.sow('intermediates', f'layer_{i+1}_output', x)
            
        # Final layer to produce output
        # x = self.classification_layer(x)
        return x


## Construction

Of the dg/dt and dg/du

In [3]:
n = 3 # 1 dimensional input and output 
L = 5 # Number of layers; trying to emulate the paper
n_samples = 2
x = jax.random.normal(jax.random.PRNGKey(0), (n_samples, n))

# First, pretend to call the model
model = SimpleMLP(num_layers=L, num_units=n, num_classes=n)
params = model.init( jax.random.PRNGKey(0), x)
predictions, intermediates = model.apply(params, x, mutable=['intermediates'])
intermediates

{'intermediates': {'layer_0_output': (<jax.Array float64(2, 3) ≈0.12 ±0.82 [≥-0.78, ≤1.8] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_1_output': (<jax.Array float64(2, 3) ≈-0.065 ±0.51 [≥-0.74, ≤0.61] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_2_output': (<jax.Array float64(2, 3) ≈-0.039 ±0.23 [≥-0.33, ≤0.23] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_3_output': (<jax.Array float64(2, 3) ≈0.027 ±0.25 [≥-0.36, ≤0.48] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_4_output': (<jax.Array float64(2, 3) ≈-0.0064 ±0.074 [≥-0.11, ≤0.085] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_5_output': (<jax.Array float64(2, 3) ≈-0.0027 ±0.087 [≥-0.094, ≤0.1] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,)}}

Need derivative of layers wrt to both inputs and weights. 

$K_i$ is the gradient wrt to parameters while $M_i$ is gradient wrt to $x$s. 

In [4]:
# Create layer model; then be able to take the gradients and stuff
layer = MLP(num_units=n)

# First define function
def apply_layer(params, x): 
    # assert len(x) == 1
    # return jnp.squeeze(layer.apply(params, x))
    return layer.apply(params, x)

# The issue with jax.grad is it doesn't automatically vectorize over batch dimension, so we do it manually with vmap and squeeze
# K_i_one = jax.grad(apply_layer, 0)
# K_i = jax.vmap(K_i_one, (None, 0)) # We don't allow vmapping over the params. I guess we could in theory... but it's easier to wrap my head around htis

# M_i_one = jax.grad(apply_layer, 1)
# M_i = jax.vmap(M_i_one, (None, 0))

# In higher batch dims and samples, we use the Jax jacobian

K_i = jax.jacrev(apply_layer, argnums=0)
M_i = jax.jacrev(apply_layer, argnums=1)

# intermediates['intermediates'][f'layer_{l}_output'][0]
# apply_layer({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
# jax.jacrev(apply_layer, argnums=0)({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
# # jax.jacrev(apply_layer, argnums=1)({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])

In [5]:
dgdu = np.eye(n_samples * n * (L + 1)) # It's tridiagonal with ones down diagonal; will have to change once scaled up
counter = 0
for l in range(L):
    # print(-jnp.squeeze(M_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])))
    # print(dgdu[l, l+1:l+n_samples+1] )
    vals = -jnp.squeeze(M_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0]))
    # print(vals.shape, vals)
    # vals
    for s in range(n_samples):
        for t in range(n):
            for u in range(n): 
                dgdu[counter * (n_samples * n) + s * n + t, 
                     counter * (n_samples * n) + (n_samples * n) + s * n + u] =  vals[s, u, s, t]
    counter += 1
    # break
dgdu.T

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(36, 36))

In [6]:
flattened, _ = jax.tree.flatten(
        K_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
)

# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
aggregated_array.shape[-1]

24

In [7]:
# Number of trainable parameters
p = sum(x.size for x in jax.tree.leaves(params))
print(f'Total parameters {p}')
dgdt = np.zeros((p, n_samples * n * (L + 1)))
# This is harder to construct; need a mapping from dof to matrix... think about this for bigger problems
# Better way would be a matrix free operation; but for now, just do 
counter = 0
for l in range(L): 
    flattened, _ = jax.tree.flatten(
            K_i({'params': params['params'][f'layer_{l:02}']}, intermediates['intermediates'][f'layer_{l}_output'][0])
    )
    
    # Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
    reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
    aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
    
    layer_p = aggregated_array.shape[-1]

    for s in range(n_samples):
        for t in range(n):
            dgdt[counter:counter + layer_p, 
                 n_samples * n * l + n_samples * n + s * n + t] =  aggregated_array[s, t]
    # counter += 1


    # Can make this dynamic 
    # dgdt[counter:counter + layer_p, n_samples * l + n_samples:n_samples * (l + 1) + n_samples] = vals

    counter += layer_p
    
dgdt


Total parameters 120


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.09711334, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.09711334,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.09711334]], shape=(120, 36))

In [8]:
derivative = np.linalg.inv(dgdu.T) @ dgdt.T
np.linalg.inv(dgdu.T)
derivative

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.05588373, -0.01938628,  0.02274723, ...,  0.09711334,
         0.        ,  0.        ],
       [-0.05137943, -0.03059333, -0.0025699 , ...,  0.        ,
         0.09711334,  0.        ],
       [ 0.05990951,  0.02072117, -0.02449751, ...,  0.        ,
         0.        ,  0.09711334]], shape=(36, 120))

In [9]:
# Let's compare this with the Jacobian 
jacobian = jax.jacobian(model.apply, argnums=0)(params, x)

flattened, _ = jax.tree.flatten(
    jacobian
)

# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
reshaped_leaves = [leaf.reshape(n_samples, n, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)


# jacobian = jnp.array([jnp.squeeze(x) for x in jax.tree.flatten(jacobian)[0]])
aggregated_array.reshape((n * n_samples, -1))

<jax.Array float32(6, 120) ≈0.0095 ±0.26 [≥-1.3, ≤1.4] zero:48 nonzero:672
  <Arrayviz rendering>
| Device: GPU 0>

In [10]:
derivative[-n_samples * n:, :]

array([[ 1.45680433e-03,  4.20820373e-02,  4.60116361e-02,
        -2.99871746e-04, -8.66225656e-03, -9.47113350e-03,
        -1.14325017e-03, -3.30245429e-02, -3.61083572e-02,
         2.64568316e-03,  7.64246267e-02,  8.35611200e-02,
        -2.93225622e-02,  8.13683746e-03,  5.96819488e-02,
        -2.87808944e-02,  7.98652785e-03,  5.85794602e-02,
        -4.40196435e-03,  1.22151906e-03,  8.95957897e-03,
        -8.16018232e-05,  2.26440230e-05,  1.66089028e-04,
         2.46231077e-01,  5.00047381e-01, -3.67934489e-01,
        -1.26165385e-01, -2.56217321e-01,  1.88524517e-01,
        -1.82512294e-01, -3.70646947e-01,  2.72721747e-01,
         1.51254906e-01,  3.07169253e-01, -2.26014915e-01,
         4.81613788e-01,  2.05434723e-01, -2.55845984e-01,
         2.85423932e-01,  1.21748978e-01, -1.51624743e-01,
         1.91550871e-01,  8.17069631e-02, -1.01756890e-01,
         3.45969874e-01,  1.47575146e-01, -1.83788349e-01,
        -1.73904799e-01,  9.05789792e-01, -2.06611811e-01,
         3.07595955e-02, -1.60212538e-01,  3.65446843e-02,
         5.75086645e-02, -2.99536123e-01,  6.83245674e-02,
         4.63384563e-02, -2.41355619e-01,  5.50535232e-02,
        -1.30574296e+00,  3.14579985e-01, -3.07816377e-01,
        -7.35625394e-02,  1.77227091e-02, -1.73416631e-02,
        -1.48938523e-02,  3.58823137e-03, -3.51108282e-03,
        -5.11331293e-01,  1.23190089e-01, -1.20541448e-01,
        -1.01582506e+00,  1.75763830e-01, -7.26229596e-01,
         1.47295430e-02, -2.54858885e-03,  1.05303855e-02,
        -1.11672032e-01,  1.93221270e-02, -7.98361195e-02,
        -4.90315394e-01,  8.48371530e-02, -3.50534328e-01,
        -6.56686300e-01,  4.22009398e-01,  2.69332672e-01,
        -3.95559012e-02,  2.54199944e-02,  1.62234184e-02,
        -1.68886880e-01,  1.08532568e-01,  6.92670985e-02,
        -7.20821115e-02,  4.63224655e-02,  2.95636862e-02,
        -9.90558267e-01, -2.46726677e-01,  6.22511923e-01,
        -5.31587340e-02, -1.32406922e-02,  3.34073678e-02,
         6.67626038e-02,  1.66291222e-02, -4.19566594e-02,
         1.10373542e-01,  2.74916664e-02, -6.93637580e-02,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         5.04657328e-02,  0.00000000e+00,  0.00000000e+00,
        -1.05169483e-01,  0.00000000e+00,  0.00000000e+00,
        -1.11512199e-01,  0.00000000e+00,  0.00000000e+00],
       [-1.27146805e-07,  8.35767088e-03,  1.39150684e-02,
         2.61675987e-08, -1.72036084e-03, -2.86430743e-03,
         9.98009771e-08, -6.55881404e-03, -1.09200695e-02,
        -2.30877902e-07,  1.51782546e-02,  2.52709708e-02,
        -7.79984149e-03,  1.64053359e-02,  1.54366123e-02,
        -7.65575712e-03,  1.61022846e-02,  1.51514559e-02,
        -1.17092852e-03,  2.46280333e-03,  2.31737652e-03,
        -2.17061962e-05,  4.56544457e-05,  4.29585826e-05,
         1.08724622e-01,  2.52961226e-01, -1.98271336e-01,
        -5.57089874e-02, -1.29613813e-01,  1.01591477e-01,
        -8.05892600e-02, -1.87500844e-01,  1.46963406e-01,
         6.67873958e-02,  1.55389098e-01, -1.21794180e-01,
         2.95198994e-01,  1.06759105e-01, -4.52372534e-02,
         1.74946938e-01,  6.32697905e-02, -2.68094375e-02,
         1.17408649e-01,  4.24609926e-02, -1.79920831e-02,
         2.12057797e-01,  7.66909815e-02, -3.24964261e-02,
        -1.47685249e-01,  2.80865958e-01, -1.70027446e-01,
         2.61219854e-02, -4.96784667e-02,  3.00737854e-02,
         4.88381101e-02, -9.28797186e-02,  5.62264644e-02,
         3.93520277e-02, -7.48391932e-02,  4.53052994e-02,
        -4.88380889e-01,  1.32039879e-01, -2.18329837e-01,
        -2.75142502e-02,  7.43882155e-03, -1.23001983e-02,
        -5.57067744e-03,  1.50610230e-03, -2.49036177e-03,
        -1.91250836e-01,  5.17070546e-02, -8.54983573e-02,
        -3.06861332e-01, -7.66731572e-02, -4.09286741e-01,
         4.44951333e-03,  1.11176695e-03,  5.93468950e-03,
        -3.37339858e-02, -8.42886092e-03, -4.49938494e-02,
        -1.48114904e-01, -3.70083730e-02, -1.9

In [11]:
derivative[-n_samples * n:, :] - aggregated_array.reshape((n * n_samples, -1))

<jax.Array float64(6, 120) ≈-9.3e-11 ±5.9e-09 [≥-4.1e-08, ≤2.9e-08] zero:144 nonzero:576
  <Arrayviz rendering>
| Device: GPU 0>

In [12]:
np.linalg.norm(derivative[-n_samples * n:, :] - aggregated_array.reshape((n * n_samples, -1))) / np.linalg.norm( aggregated_array.reshape((n * n_samples, -1)))

np.float64(2.294689923628117e-08)

## Construction of precond 

The preconditioner shouldn't change much

In [13]:
# Matrix which we want ot approximate 
jnp.linalg.inv(dgdu.T @ dgdu)

<jax.Array float64(36, 36) ≈0.035 ±0.59 [≥-2.4, ≤7.9] zero:648 nonzero:648
  <Arrayviz rendering>
| Device: GPU 0>

In [14]:
def partition_with_overlap(L, N):
    # Compute the approximate size of each partition
    step = L / N
    
    # Generate the partition points
    c = [round(i * step) for i in range(N + 1)]
    
    return c


In [19]:
def make_matrices(L, ND): 
    """
    Given L layers and ND blocks, give the restriction, pou and q matrices for testing
    """
    restrictions = []
    pous = []
    qs = []

    cs = partition_with_overlap(L, ND)
    for d in range(ND): 
        time_steps = cs[d + 1] - cs[d] + 1
        r = np.zeros((n_samples * n * time_steps, n_samples * n * (L + 1)))
        for t in range(n_samples * n * time_steps):
            r[t, n_samples * n * cs[d] + t] = 1

        pou = np.eye(n_samples * n * time_steps)
        if d == 0:
            pou[np.arange(len(pou) - n_samples * n, len(pou)), np.arange(len(pou) - n_samples * n, len(pou))] = 0.5
        elif d == ND - 1:
            pou[np.arange(0, n_samples * n), np.arange(0, n_samples* n)] = 0.5
        else: 
            pou[np.arange(0, n_samples * n), np.arange(0, n_samples* n)] = 0.5
            pou[np.arange(len(pou) - n_samples* n, len(pou)), np.arange(len(pou) - n_samples* n, len(pou))] = 0.5

        if d == 0:
            q = np.zeros((n_samples* n * time_steps, n_samples* n * (L + 1)))
            for i in range(n_samples* n * time_steps): 
                q[i, i] = 1
        else:
            q = np.zeros((n_samples* n * (time_steps + 1), n_samples* n *( L + 1)))
            for t in range(n_samples* n * (time_steps + 1)): 
                q[t, n_samples* n * cs[d] - n_samples* n + t] = 1
        
        restrictions.append(r)
        pous.append(pou)
        qs.append(q)

    # assert np.linalg.norm(R1.T @ D1 @ R1 + R2.T @ D2 @ R2 + R3.T @ D3 @ R3 - np.eye(10)) < 1e-10

    return restrictions, pous, qs


In [20]:
ND = 3
make_matrices(L, ND)

([array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.]]),
  array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,


In [29]:
restrictions, pous, qs = make_matrices(L, ND)
# Construct RAS preconditioner 
ras = np.zeros((n_samples* n* (L + 1), n_samples * n * (L + 1)))
for i in range(ND): 
    R = restrictions[i]
    D = pous[i]
    ras += R.T @ D @ jnp.linalg.inv(R @ dgdu.T @ dgdu @ R.T) @ R


In [30]:
ras

<jax.Array float64(36, 36) ≈0.031 ±0.28 [≥-1.4, ≤2.7] zero:936 nonzero:360
  <Arrayviz rendering>
| Device: GPU 0>

In [31]:
    print(np.linalg.eigvals(ras @  dgdu.T @ dgdu))


[1.        +0.j         0.26304398+0.j         0.54602297+0.j
 0.2536143 +0.j         1.345824  +0.13748856j 1.345824  -0.13748856j
 0.47056141+0.j         1.36973486+0.j         0.87066075+0.j
 1.3430233 +0.13896882j 1.3430233 -0.13896882j 1.36124025+0.j
 1.08911026+0.j         1.06932557+0.0448205j  1.06932557-0.0448205j
 0.86907657+0.j         1.15722333+0.j         1.02643555+0.j
 1.00468091+0.j         1.07658829+0.0304725j  1.07658829-0.0304725j
 1.03098959+0.j         1.01806501+0.j         1.00001157+0.j
 1.        +0.j         1.        +0.j         1.00000637+0.j
 1.        +0.j         1.        +0.j         1.        +0.j
 1.        +0.j         1.        +0.j         1.        +0.j
 1.        +0.j         1.        +0.j         1.        +0.j        ]


In [20]:
rasq = np.zeros((n_samples * (L + 1), n_samples * (L + 1)))
for i in range(ND): 
    R = restrictions[i]
    D = pous[i]
    Q = qs[i]
    Js = np.linalg.inv(Q @ dgdu @ Q.T)
    rasq += R.T @ D @ (R @ Q.T @ Js @ Js.T @ Q @ R.T) @ R
print(np.linalg.eigvals(rasq @  dgdu.T @ dgdu))
rasq

[15.94478747+0.00000000e+00j  0.11376382+0.00000000e+00j
  1.16640995+0.00000000e+00j  6.17192651+0.00000000e+00j
  0.87918445+0.00000000e+00j  3.41479168+0.00000000e+00j
  0.121243  +0.00000000e+00j  0.85271922+0.00000000e+00j
  1.        +7.28021923e-16j  1.        -7.28021923e-16j
  1.        +1.33688556e-15j  1.        -1.33688556e-15j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j
  1.        +0.00000000e+00j  1.        +0.00000000e+00j]


array([[9.18131109, 0.        , 4.06820147, 0.        , 3.34403414,
        0.        , 1.61447263, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 4.75030661, 0.        , 2.77907167, 0.        ,
        2.00487751, 0.        , 1.00163307, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [4.06820147, 0.        , 2.02293533, 0.        , 1.66283918,
        0.        , 0.8028053 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 2.77907167, 0.        , 2.05936212, 0.        ,
        1.48566474, 0.        , 0.74223534, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [3.34403414, 0.        , 1.66283918, 0.        , 2.70303905,
        0.        , 1.30500538, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 2.00487751, 0.        , 1.48566474, 0.        ,
        2.0835177 , 0.        , 1.04092156, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.80723631, 0.        , 0.40140265, 0.        , 0.65250269,
        0.        , 8.75568129, 0.        , 2.26990544, 0.        ,
        0.984012  , 0.        , 0.09427282, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.50081654, 0.        , 0.37111767, 0.        ,
        0.52046078, 0.        , 3.00482951, 0.        , 1.42472781,
        0.        , 0.96269391, 0.        , 0.47917259, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 4.53981089, 0.        , 1.32869584, 0.        ,
        0.57599433, 0.        , 0.05518288, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 2.84945561, 0.        , 2.02495954,
        0.        , 1.36827274, 0.        , 0.68104595, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 1.96802401, 0.        , 0.57599433, 0.        ,
        1.00935097, 0.        , 0.09670041, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 1.92538783, 0.        , 1.36827274,
        0.        , 1.82657971, 0.        , 0.90916429, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.09427282, 0.        , 0.02759144, 0.        ,
        0.04835021, 0.        , 5.24796368, 0.        , 3.20152702,
        0.        , 2.53947092, 0.        , 1.11543016, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.47917259, 0.        , 0.34052298,
        0.        , 0.45458215, 0.        , 5.15369917, 0.        ,
        3.13739817, 0.        , 2.47677037, 0.        , 1.09586768],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 6.40305404

## Need to do multi-dimensional
